## Introducing new bias tools

### 1. Import Samna

Import Samna and all other necessary packages

In [1]:
import samna
import samna.dynapse1 as dyn1

import sys
sys.path.append("..")


from dynapse1constants import *

import dynapse1utils as ut
import netgen as n
from netgen import Neuron
import time
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# open without GUI (for board connected to the remote machine)
model, _ = ut.open_dynapse1(gui=True, select_device=True)

# open with GUI (only with the board connected locally)
#model, gui_thread = ut.open_dynapse1(gui=True)

[0]:  Bus 3 Device 10 Dynapse1DevKit serial_number 00000017


Select the device you want to open by index:  0


Visualizer start command:  /home/giacomo/miniconda3/envs/dynapse/bin/python -c "import samna, samnagui; samnagui.runVisualizer(0.75, 0.75,         'tcp://0.0.0.0:33335', 'tcp://0.0.0.0:33336', 3)"


Traceback (most recent call last):
  File "<string>", line 1, in <module>
RuntimeError: glfwCreateWindow(...) failed! Error code: 65545 description: GLX: Failed to find a suitable GLXFBConfig



Exception: open_remote_node failed:  visualizer id 3 can't be opened in 10 seconds!!

#### 1. Unifying ways to set biases. "Linear", "index" and "tuple"

The parameters can now be set and retrieved using all three formats.

**linear** - value from 0 to 24e+08, proportional to value in pA, estimated from the bias generator model. Any relations between biases (e.g. clamping I_gain to I_tau) can be done using this format.

**tuple** - the coarse [0, 7] and fine [0, 255] pair. This format can still be used by those who prefer it, while it is less intuitive.

**index** - value from 0 to 1811 which indexes the map between **linear** and **tuple** values.

**Dynapse1utils** now contains the converter functions that convert those formats one to another.

Note that the **linear** bias is rounded to the nearest _valid_ value that actually corresponds to a tuple of coarse and fine.

In [ ]:
example_tuple = (3, 100)
example_index = 105
example_linear = 578645

print("Examples of index values:",
      ut._bias_as_idx(example_tuple, "tuple"),
      ut._bias_as_idx(example_index, "index"),
      ut._bias_as_idx(example_linear, "linear"))

print("Examples of tuple values:",
      ut._bias_as_tuple(example_tuple, "tuple"),
      ut._bias_as_tuple(example_index, "index"),
      ut._bias_as_tuple(example_linear, "linear"))

print("Examples of linear value:",
      ut._bias_as_linear(example_tuple, "tuple"),
      ut._bias_as_linear(example_index, "index"),
      ut._bias_as_linear(example_linear, "linear"))

Parameters can be set directly to the cores using **named functions**:

In [ ]:
# set parameter as index
ut.set_neuron_dc(model, chip_id=0, core_id=0, bias_value=50, bias_format='index')


# set parameter as a tuple
ut.set_neuron_tau1(model, chip_id=0, core_id=0, bias_value=(4, 100), bias_format='tuple')

In [ ]:
print("C1c0 refractory period bias in all formats:",
      ut.get_neuron_refractory_period(model, chip_id=1, core_id=0, bias_format="tuple"),
      ut.get_neuron_refractory_period(model, chip_id=1, core_id=0, bias_format="index"),
      ut.get_neuron_refractory_period(model, chip_id=1, core_id=0, bias_format="linear"))

Clamping I_gain to I_tau is now possible. Regardless of the format given, the factor is applied to the **linear** value of the bias:

In [ ]:
bias_value_linear = 10000
gain_ratio = 4

ut.set_ampa_tau(model, chip_id=0, core_id=0, bias_value=bias_value_linear, bias_format='linear', fixed_gain_ratio=gain_ratio)

print("C0c0 AMPA tau as linear:",
      ut.get_ampa_tau(model, chip_id=0, core_id=0, bias_format="linear"))
print("C0c0 AMPA gain as linear:",
      ut.get_ampa_gain(model, chip_id=0, core_id=0, bias_format="linear"))

In [ ]:
import importlib
import dynapse1utils
importlib.reload(dynapse1utils)

import dynapse1utils as ut

### 2. Setting and saving parameters of a single core

The saved parameter set can be detached from the core and therefore can be loaded to multiple cores at once easily.
The file can also be incomplete and only contain some biases.
Short descriptions for parameter values are also included in the file by default.

In [ ]:
config = model.get_configuration()

ut.save_param_group_to_txt_file(config, 1, 0, "example_core_biases.txt")


For convenience of iteratively modifying parameters in the file, the priority bias format argument is introduced. For example, one can only modify the **linear** values in the saved file, and other values would then be ignored. The function would only raise warning if the parameters are inconsistent.

Once done with tuning, simply overwrite the file with coherent values using the function above.

In [ ]:
ut.set_param_group_from_txt_file(model, 0, 0, "example_core_biases.txt", priority_bias_format='linear')

TO ADD: A few example parameter files that work for most of the cores.

### 3. Notebook slider builder

Now one can get sliders directly in Jupyter Notebooks and only for the parameters needed by passing a dictionary of **tags** with the `CXcx:` prefix indicating the chip and core.

Full list of tags is stored in `dynapse1constans.BIAS_TAGS`.

Once created, these sliders interactively update respective parameters.

In [4]:
import live_gui.notebook_gui_tools as gt

selected_bias_tags = ['C0c0:tau1',
                      'C0c0:ampa_w',
                      'C0c1:ampa_w']

gt.make_parameter_slider_box(model, selected_bias_tags, "My Network Parameters")

ModuleNotFoundError: No module named 'spikegen_tools'

### 4. Dynapse Tuning Companion

This is a helper class that provides input to the selected core and helps with monitoring.

In [24]:
from dynapse_tuning_companion import DynapseTuningCompanion
import netgen

net_gen = netgen.NetworkGenerator()
DTC = DynapseTuningCompanion(model, net_gen)

NameError: name 'target_chip_id' is not defined

In [ ]:
DTC.monitor_neuron(1, 100)

In [ ]:
DTC.connect_spikegen(target_chip_id=1)

In [ ]:
DTC.set_spikegen_rate(50)

In [ ]:
# sweep weight, threshold and voltage peaks



